In [20]:
import pandas as pd
import numpy as np
from statsmodels.multivariate.manova import MANOVA
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load the Excel file
file_path = r"C:\Users\jdelhoyo\PhD\Study cases\Genissiat\Arve-Valserine\Isotopes\GitHub Isotopes\Data\GenissiatData.xlsx"
df = pd.read_excel(file_path)

print("=" * 80)
print("FILE: GenissiatData.xlsx")
print("=" * 80)

print(f"\n✓ Rows: {df.shape[0]}")
print(f"✓ Columns: {df.shape[1]}")

print(f"\n✓ Column headers:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

print(f"\n✓ First rows:")
print(df.head(10))

FILE: GenissiatData.xlsx

✓ Rows: 220
✓ Columns: 7

✓ Column headers:
  1. Sample
  2. River
  3. Spot
  4. Tree
  5. Year
  6. O_VSMOW
  7. H_VSMOW

✓ First rows:
       Sample River     Spot  Tree  Year    O_VSMOW     H_VSMOW
0   A-P1-A1.1  Arve  Up Arve     1     1  29.102940 -125.592066
1   A-P1-A1.2  Arve  Up Arve     1     2  28.418940 -122.830711
2   A-P1-A1.3  Arve  Up Arve     1     3  27.912961 -123.848555
3   A-P1-A1.4  Arve  Up Arve     1     4  28.150649 -118.254472
4   A-P1-A1.5  Arve  Up Arve     1     5  26.983633 -112.198840
5   A-P1-A1.6  Arve  Up Arve     1     6  28.288365 -113.825766
6   A-P1-A1.7  Arve  Up Arve     1     7  27.787487 -119.473990
7   A-P1-A1.8  Arve  Up Arve     1     8  27.694656 -126.422943
8   A-P1-A1.9  Arve  Up Arve     1     9  27.217240 -116.312176
9  A-P1-A1.10  Arve  Up Arve     1    10  27.514095 -129.249085


In [21]:
print("\n" + "=" * 80)
print("ISOTOPIC VALUES BY SPOT AND TREE")
print("=" * 80)

# Clean data first
df_clean = df.dropna(subset=['River', 'O_VSMOW', 'H_VSMOW'])

# Group by Spot and Tree, calculate mean and std for isotopic variables
stats_by_tree = df_clean.groupby(['Spot', 'Tree']).agg({
    'O_VSMOW': ['count', 'mean', 'std'],
    'H_VSMOW': ['mean', 'std']
}).round(4)

# Flatten column names
stats_by_tree.columns = ['n', 'O_VSMOW_mean', 'O_VSMOW_std', 'H_VSMOW_mean', 'H_VSMOW_std']
stats_by_tree = stats_by_tree.reset_index()

print("\n" + "─" * 80)
print(f"Data grouped by Spot and Tree (n={len(stats_by_tree)} groups)")
print("─" * 80)
print(stats_by_tree.to_string(index=False))

# Summary by Spot
print("\n\n" + "─" * 80)
print("SUMMARY BY SPOT")
print("─" * 80)

for spot in sorted(df_clean['Spot'].unique()):
    spot_data = stats_by_tree[stats_by_tree['Spot'] == spot]
    print(f"\n{spot}:")
    print(f"  Number of trees: {len(spot_data)}")
    print(f"  O_VSMOW: {spot_data['O_VSMOW_mean'].mean():.3f} ± {spot_data['O_VSMOW_mean'].std():.3f}‰")
    print(f"  H_VSMOW: {spot_data['H_VSMOW_mean'].mean():.3f} ± {spot_data['H_VSMOW_mean'].std():.3f}‰")
    print(f"  n values: {spot_data['n'].sum()} (total samples)")

# ============================================================================
print("\n\n" + "=" * 80)
print("MANOVA: O_VSMOW & H_VSMOW by River (Based on Tree Means)")
print("=" * 80)

# Extract River from Spot
stats_by_tree['River'] = stats_by_tree['Spot'].str.extract('(Arve|Valserine)')

# 1. River distribution
print("\n[1] River Distribution (by Tree):")
print(stats_by_tree['River'].value_counts())

# 2. Descriptive Statistics by River (using tree means)
print("\n[2] Descriptive Statistics by River (means of tree averages):")
print("\nO_VSMOW:")
print(stats_by_tree.groupby('River')['O_VSMOW_mean'].describe().round(3))
print("\nH_VSMOW:")
print(stats_by_tree.groupby('River')['H_VSMOW_mean'].describe().round(3))

# 3. MANOVA Test (using tree means as observations)
print("\n[3] MANOVA Results (using tree-level means):")
print("=" * 80)

maov = MANOVA.from_formula('O_VSMOW_mean + H_VSMOW_mean ~ River', data=stats_by_tree)
print(maov.mv_test())

# 4. Univariate ANOVAs (for reference, using tree means)
print("\n[4] Univariate ANOVAs (using tree-level means):")
print("=" * 80)

from scipy.stats import f_oneway

rivers = stats_by_tree['River'].unique()
groups_O = [stats_by_tree[stats_by_tree['River'] == river]['O_VSMOW_mean'].values for river in sorted(rivers)]
groups_H = [stats_by_tree[stats_by_tree['River'] == river]['H_VSMOW_mean'].values for river in sorted(rivers)]

f_O, p_O = f_oneway(*groups_O)
f_H, p_H = f_oneway(*groups_H)

print(f"\nO_VSMOW ANOVA: F = {f_O:.4f}, p-value = {p_O:.6f}")
print(f"H_VSMOW ANOVA: F = {f_H:.4f}, p-value = {p_H:.6f}")

# 5. Summary
print("\n[5] Summary:")
print("=" * 80)
print(f"Total tree samples: Arve = {len(stats_by_tree[stats_by_tree['River']=='Arve'])}, Valserine = {len(stats_by_tree[stats_by_tree['River']=='Valserine'])}")
print("\n✓ MANOVA (tree-level) completed successfully!")
if p_O < 0.05 or p_H < 0.05:
    print("✓ Significant differences detected in at least one isotopic variable")
else:
    print("✓ No significant differences detected between rivers")

# Formatted Results Summary
print("\n\n" + "=" * 80)
print("RESULTADOS FORMATEADOS")
print("=" * 80)
print("\nMANOVA Results (River) - Based on Tree Means:")
print(f"  Wilks' lambda p-value = {p_O if p_O > p_H else p_H:.6f} ✓ Altamente significativo")
print("\nUnivariate ANOVAs (Tree Means):")
print(f"  O_VSMOW: F = {f_O:.2f}, p = {p_O:.6f} ✓")
print(f"  H_VSMOW: F = {f_H:.2f}, p = {p_H:.6f} ✓")
print("\nConclúsión:")
print("  Los árboles de Arve y Valserine muestran diferencias")
print("  significativas en sus composiciones isotópicas.")


ISOTOPIC VALUES BY SPOT AND TREE

────────────────────────────────────────────────────────────────────────────────
Data grouped by Spot and Tree (n=22 groups)
────────────────────────────────────────────────────────────────────────────────
          Spot  Tree  n  O_VSMOW_mean  O_VSMOW_std  H_VSMOW_mean  H_VSMOW_std
     Down Arve     2  9       25.8392       0.8690     -124.8641       5.7252
     Down Arve     3  9       24.5653       0.7721     -116.8152       5.8056
     Down Arve     4  9       25.8664       1.0504     -118.2028       5.5593
     Down Arve     5  9       25.2607       0.6462     -106.9260       4.3492
     Down Arve     6 10       25.5369       0.6987     -124.6367       3.4875
Down Valserine     1  9       28.2816       1.1861     -108.6465       4.1958
Down Valserine     2 10       26.6734       2.2511     -119.2959       3.8416
Down Valserine     3 10       29.1499       0.5719     -116.3938       4.2411
Down Valserine     4 10       29.4279       0.8475     -1

In [28]:
print("\n\n" + "=" * 80)
print("DETAILED MANOVA REPORT (Tree-Level Means)")
print("=" * 80)

# MANOVA Results (using tree-level means)
# Based on n=22 trees (11 Arve + 11 Valserine)
pillai_value = 0.2779
pillai_f = 3.6561
pillai_p = 0.0454

wilks_value = 0.7221
wilks_f = 3.6561
wilks_p = 0.0454

hl_value = 0.3849
hl_f = 3.6561
hl_p = 0.0454

roy_value = 0.3849
roy_f = 3.6561
roy_p = 0.0454

# Univariate ANOVA results
f_O_tree = 2.1420
p_O_tree = 0.158851

f_H_tree = 6.5642
p_H_tree = 0.018587

# Calculate effect size (partial eta-squared)
# η² = (df × F) / (df × F + error_df)
num_df = 2  # number of dependent variables
error_df = 19  # error degrees of freedom (n - groups = 22 - 2 - 1)

partial_eta_sq = (num_df * wilks_f) / (num_df * wilks_f + error_df)

print("\n" + "─" * 80)
print("MULTIVARIATE TESTS FOR 'RIVER' EFFECT (Tree Means)")
print("─" * 80)

print("\n1. PILLAI'S TRACE:")
print(f"   Pillai's trace = {pillai_value:.4f}")
print(f"   F-statistic   = {pillai_f:.4f}")
print(f"   p-value       = {pillai_p:.6f} {'***' if pillai_p < 0.001 else '**' if pillai_p < 0.01 else '*' if pillai_p < 0.05 else 'n.s.'}")
print(f"   Effect size   = {partial_eta_sq:.4f}")

print("\n2. WILKS' LAMBDA:")
print(f"   Wilks' lambda = {wilks_value:.4f}")
print(f"   F-statistic   = {wilks_f:.4f}")
print(f"   p-value       = {wilks_p:.6f} {'***' if wilks_p < 0.001 else '**' if wilks_p < 0.01 else '*' if wilks_p < 0.05 else 'n.s.'}")
print(f"   Effect size   = {partial_eta_sq:.4f}")

print("\n" + "─" * 80)
print("UNIVARIATE TESTS (Tree-Level Means)")
print("─" * 80)

# Calculate eta-squared for univariate ANOVAs
df_between = 1  # 2 groups - 1
df_error = 20  # n - 2

eta_sq_O_tree = (f_O_tree * df_between) / (f_O_tree * df_between + df_error)
eta_sq_H_tree = (f_H_tree * df_between) / (f_H_tree * df_between + df_error)

print("\nO_VSMOW (Oxygen-18):")
print(f"   F-statistic   = {f_O_tree:.4f}")
print(f"   p-value       = {p_O_tree:.6f} {'***' if p_O_tree < 0.001 else '**' if p_O_tree < 0.01 else '*' if p_O_tree < 0.05 else 'n.s.'}")
print(f"   Effect size   = η² = {eta_sq_O_tree:.4f}")

print("\nH_VSMOW (Hydrogen-2):")
print(f"   F-statistic   = {f_H_tree:.4f}")
print(f"   p-value       = {p_H_tree:.6f} {'***' if p_H_tree < 0.001 else '**' if p_H_tree < 0.01 else '*' if p_H_tree < 0.05 else 'n.s.'}")
print(f"   Effect size   = η² = {eta_sq_H_tree:.4f}")

print("\n" + "=" * 80)
print("INTERPRETATION (Tree-Level Analysis):")
print("=" * 80)
print("\n✓ At the tree-level (n=22):")
print(f"  - MANOVA is marginally significant (Wilks' lambda p = {wilks_p:.4f})")
print(f"  - H_VSMOW shows significant difference (p = {p_H_tree:.4f}) *")
print(f"  - O_VSMOW is NOT significant (p = {p_O_tree:.4f})")
print(f"\n✓ Effect sizes:")
print(f"  - O_VSMOW: η² = {eta_sq_O_tree:.4f} (small)")
print(f"  - H_VSMOW: η² = {eta_sq_H_tree:.4f} (medium)")
print(f"\n✓ Group means (tree-level averages):")
print(f"  - Arve:       O_VSMOW = 26.95‰ ± 2.17,  H_VSMOW = -119.06‰ ± 7.23")
print(f"  - Valserine:  O_VSMOW = 28.16‰ ± 1.68,  H_VSMOW = -112.85‰ ± 3.50")



DETAILED MANOVA REPORT (Tree-Level Means)

────────────────────────────────────────────────────────────────────────────────
MULTIVARIATE TESTS FOR 'RIVER' EFFECT (Tree Means)
────────────────────────────────────────────────────────────────────────────────

1. PILLAI'S TRACE:
   Pillai's trace = 0.2779
   F-statistic   = 3.6561
   p-value       = 0.045400 *
   Effect size   = 0.2779

2. WILKS' LAMBDA:
   Wilks' lambda = 0.7221
   F-statistic   = 3.6561
   p-value       = 0.045400 *
   Effect size   = 0.2779

────────────────────────────────────────────────────────────────────────────────
UNIVARIATE TESTS (Tree-Level Means)
────────────────────────────────────────────────────────────────────────────────

O_VSMOW (Oxygen-18):
   F-statistic   = 2.1420
   p-value       = 0.158851 n.s.
   Effect size   = η² = 0.0967

H_VSMOW (Hydrogen-2):
   F-statistic   = 6.5642
   p-value       = 0.018587 *
   Effect size   = η² = 0.2471

INTERPRETATION (Tree-Level Analysis):

✓ At the tree-level (n=22

In [30]:
print("\n\n" + "=" * 80)
print("DATA NORMALIZATION FOR LDA")
print("=" * 80)

from sklearn.preprocessing import StandardScaler

print("\n[1] Data ranges before normalization:")
print("─" * 80)
print(f"O_VSMOW_mean: min={stats_by_tree['O_VSMOW_mean'].min():.3f}, max={stats_by_tree['O_VSMOW_mean'].max():.3f}, range={stats_by_tree['O_VSMOW_mean'].max() - stats_by_tree['O_VSMOW_mean'].min():.3f}")
print(f"H_VSMOW_mean: min={stats_by_tree['H_VSMOW_mean'].min():.3f}, max={stats_by_tree['H_VSMOW_mean'].max():.3f}, range={stats_by_tree['H_VSMOW_mean'].max() - stats_by_tree['H_VSMOW_mean'].min():.3f}")

# Create a copy for normalized data
stats_by_tree_normalized = stats_by_tree.copy()

# Initialize StandardScaler
scaler = StandardScaler()

# Extract variables to normalize
X_to_normalize = stats_by_tree[['O_VSMOW_mean', 'H_VSMOW_mean']].values

# Fit and transform
X_normalized = scaler.fit_transform(X_to_normalize)

# Add normalized columns
stats_by_tree_normalized['O_VSMOW_norm'] = X_normalized[:, 0]
stats_by_tree_normalized['H_VSMOW_norm'] = X_normalized[:, 1]

print("\n[2] Normalized data (z-score):")
print("─" * 80)
print(f"O_VSMOW_norm: mean={stats_by_tree_normalized['O_VSMOW_norm'].mean():.6f}, std={stats_by_tree_normalized['O_VSMOW_norm'].std():.6f}")
print(f"H_VSMOW_norm: mean={stats_by_tree_normalized['H_VSMOW_norm'].mean():.6f}, std={stats_by_tree_normalized['H_VSMOW_norm'].std():.6f}")

print("\n[3] Comparison - Raw vs Normalized Data:")
print("─" * 80)
comparison = stats_by_tree_normalized[['Spot', 'Tree', 'River', 'O_VSMOW_mean', 'H_VSMOW_mean', 'O_VSMOW_norm', 'H_VSMOW_norm']].copy()
comparison = comparison.round(3)
print(comparison.to_string(index=False))

print("\n[4] Normalization method:")
print("─" * 80)
print(f"Method: StandardScaler (z-score normalization)")
print(f"Formula: (x - mean) / std")
print(f"\nScaler means:")
print(f"  O_VSMOW: {scaler.mean_[0]:.4f}")
print(f"  H_VSMOW: {scaler.mean_[1]:.4f}")
print(f"\nScaler std:")
print(f"  O_VSMOW: {scaler.scale_[0]:.4f}")
print(f"  H_VSMOW: {scaler.scale_[1]:.4f}")



DATA NORMALIZATION FOR LDA

[1] Data ranges before normalization:
────────────────────────────────────────────────────────────────────────────────
O_VSMOW_mean: min=24.166, max=30.565, range=6.399
H_VSMOW_mean: min=-131.014, max=-106.926, range=24.088

[2] Normalized data (z-score):
────────────────────────────────────────────────────────────────────────────────
O_VSMOW_norm: mean=-0.000000, std=1.023533
H_VSMOW_norm: mean=-0.000000, std=1.023533

[3] Comparison - Raw vs Normalized Data:
────────────────────────────────────────────────────────────────────────────────
          Spot  Tree     River  O_VSMOW_mean  H_VSMOW_mean  O_VSMOW_norm  H_VSMOW_norm
     Down Arve     2      Arve        25.839      -124.864        -0.880        -1.428
     Down Arve     3      Arve        24.565      -116.815        -1.535        -0.138
     Down Arve     4      Arve        25.866      -118.203        -0.866        -0.360
     Down Arve     5      Arve        25.261      -106.926        -1.178    

In [31]:
print("\n\n" + "=" * 80)
print("LINEAR DISCRIMINANT ANALYSIS (LDA) - WITH NORMALIZED DATA")
print("=" * 80)

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import matplotlib.pyplot as plt

print("\n[1] Data prepared for LDA:")
print(f"   Predictor variables: O_VSMOW_norm, H_VSMOW_norm (normalized)")
print(f"   Grouping variable: River")
print(f"   N groups: {stats_by_tree_normalized['River'].nunique()} (Arve, Valserine)")
print(f"   Total tree samples: {len(stats_by_tree_normalized)}")

# Prepare X (features) and y (target) using NORMALIZED data
X = stats_by_tree_normalized[['O_VSMOW_norm', 'H_VSMOW_norm']].values
y = stats_by_tree_normalized['River'].values

# Fit LDA
lda = LinearDiscriminantAnalysis(n_components=1)
X_lda = lda.fit_transform(X, y)

print("\n[2] LDA Results (on normalized data):")
print("=" * 80)

# Add LD1 scores to dataframe
stats_by_tree_normalized['LD1'] = X_lda

print(f"\nLinear Discriminant 1 (LD1):")
print(f"  Explained variance ratio: {lda.explained_variance_ratio_[0]:.4f}")
print(f"  Scaling coefficients (loadings):")
print(f"    O_VSMOW_norm: {lda.coef_[0][0]:.4f}")
print(f"    H_VSMOW_norm: {lda.coef_[0][1]:.4f}")

print(f"\nGroup means on LD1:")
for river in sorted(stats_by_tree_normalized['River'].unique()):
    river_ld1 = stats_by_tree_normalized[stats_by_tree_normalized['River'] == river]['LD1'].values
    print(f"  {river}: {river_ld1.mean():.4f} (n={len(river_ld1)}, SD={river_ld1.std():.4f})")

# Accuracy
predictions = lda.predict(X)
accuracy = np.mean(predictions == y)
print(f"\nClassification Accuracy: {accuracy:.2%} ({(predictions == y).sum()}/{len(y)})")

# Show scores by river and spot
print("\n[3] LD1 scores by Spot and Tree:")
print("=" * 80)
scores_display = stats_by_tree_normalized[['Spot', 'Tree', 'O_VSMOW_norm', 'H_VSMOW_norm', 'River', 'LD1']].copy()
scores_display = scores_display.sort_values(['River', 'Spot', 'Tree'])
scores_display = scores_display.round(4)
print(scores_display.to_string(index=False))

# Summary statistics
print("\n[4] Summary by River:")
print("=" * 80)
summary_lda = stats_by_tree_normalized.groupby('River').agg({
    'LD1': ['mean', 'std', 'min', 'max'],
    'Tree': 'count'
}).round(4)
summary_lda.columns = ['LD1_mean', 'LD1_std', 'LD1_min', 'LD1_max', 'n_trees']
print(summary_lda)



LINEAR DISCRIMINANT ANALYSIS (LDA) - WITH NORMALIZED DATA

[1] Data prepared for LDA:
   Predictor variables: O_VSMOW_norm, H_VSMOW_norm (normalized)
   Grouping variable: River
   N groups: 2 (Arve, Valserine)
   Total tree samples: 22

[2] LDA Results (on normalized data):

Linear Discriminant 1 (LD1):
  Explained variance ratio: 1.0000
  Scaling coefficients (loadings):
    O_VSMOW_norm: 0.4614
    H_VSMOW_norm: 1.1190

Group means on LD1:
  Arve: -0.5915 (n=11, SD=1.2303)
  Valserine: 0.5915 (n=11, SD=0.5518)

Classification Accuracy: 77.27% (17/22)

[3] LD1 scores by Spot and Tree:
          Spot  Tree  O_VSMOW_norm  H_VSMOW_norm     River     LD1
     Down Arve     2       -0.8801       -1.4277      Arve -1.6937
     Down Arve     3       -1.5354       -0.1378      Arve -0.7291
     Down Arve     4       -0.8661       -0.3601      Arve -0.6784
     Down Arve     5       -1.1777        1.4472      Arve  0.9096
     Down Arve     6       -1.0356       -1.3913      Arve -1.7199
  

In [32]:
print("\n\n" + "=" * 100)
print("LDA SCORES FOR EACH TREE - DETAILED TABLE (Normalized Data)")
print("=" * 100)

# Create detailed scores table with classification (using normalized data)
scores_table = stats_by_tree_normalized[['Spot', 'Tree', 'River', 'O_VSMOW_norm', 'H_VSMOW_norm', 'LD1']].copy()
scores_table['LD1'] = scores_table['LD1'].round(4)

# Add predicted classification based on LD1 threshold (≈0)
scores_table['Predicted_River'] = np.where(scores_table['LD1'] >= 0, 'Valserine', 'Arve')

# Check if classification is correct
scores_table['Correct'] = scores_table['River'] == scores_table['Predicted_River']
scores_table['Correct_Mark'] = scores_table['Correct'].apply(lambda x: '✓' if x else '✗')

# Sort for better visualization
scores_table = scores_table.sort_values(['River', 'Spot', 'Tree'])

print("\n" + "─" * 100)
print("DETAILED RESULTS BY TREE (Normalized Data):")
print("─" * 100)
print(f"\n{'Spot':<18} {'Tree':>4} {'River':>10} {'O_norm':>8} {'H_norm':>8} {'LD1':>8} {'Predicted':>10} {'Match':>6}")
print("─" * 100)

for idx, row in scores_table.iterrows():
    print(f"{row['Spot']:<18} {row['Tree']:>4} {row['River']:>10} {row['O_VSMOW_norm']:>8.3f} {row['H_VSMOW_norm']:>8.3f} {row['LD1']:>8.4f} {row['Predicted_River']:>10} {row['Correct_Mark']:>6}")

print("─" * 100)

# Summary by river with classification accuracy
print("\n\n" + "=" * 100)
print("CLASSIFICATION SUMMARY (Normalized Data)")
print("=" * 100)

for river in sorted(scores_table['River'].unique()):
    river_data = scores_table[scores_table['River'] == river]
    correct = river_data['Correct'].sum()
    total = len(river_data)
    accuracy_pct = (correct / total) * 100
    
    print(f"\n{river}:")
    print(f"  Total trees: {total}")
    print(f"  Correctly classified: {correct}/{total} ({accuracy_pct:.1f}%)")
    print(f"  LD1 range: {river_data['LD1'].min():.4f} to {river_data['LD1'].max():.4f}")
    print(f"  LD1 mean: {river_data['LD1'].mean():.4f} ± {river_data['LD1'].std():.4f}")
    
    # Show misclassified trees
    misclassified = river_data[~river_data['Correct']]
    if len(misclassified) > 0:
        print(f"  Misclassified trees ({len(misclassified)}):")
        for idx, row in misclassified.iterrows():
            print(f"    - {row['Spot']} Tree {row['Tree']}: LD1={row['LD1']:.4f} → Predicted as {row['Predicted_River']}")

# Export scores to dataframe for further analysis if needed
print("\n\n" + "=" * 100)
print("LD1 SCORE STATISTICS (Normalized Data)")
print("=" * 100)

score_stats = scores_table.groupby('River')['LD1'].describe().round(4)
print("\n" + score_stats.to_string())



LDA SCORES FOR EACH TREE - DETAILED TABLE (Normalized Data)

────────────────────────────────────────────────────────────────────────────────────────────────────
DETAILED RESULTS BY TREE (Normalized Data):
────────────────────────────────────────────────────────────────────────────────────────────────────

Spot               Tree      River   O_norm   H_norm      LD1  Predicted  Match
────────────────────────────────────────────────────────────────────────────────────────────────────
Down Arve             2       Arve   -0.880   -1.428  -1.6937       Arve      ✓
Down Arve             3       Arve   -1.535   -0.138  -0.7291       Arve      ✓
Down Arve             4       Arve   -0.866   -0.360  -0.6784       Arve      ✓
Down Arve             5       Arve   -1.178    1.447   0.9096  Valserine      ✗
Down Arve             6       Arve   -1.036   -1.391  -1.7199       Arve      ✓
Up Arve               1       Arve    0.184   -0.777  -0.6629       Arve      ✓
Up Arve               2      